# Tarea 18 · Heatmap de correlación

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/julianschmidt1/tpi-analisis-de-datos/blob/main/notebooks/trabajo/18_heatmap_correlacion.ipynb)

**TPI · Introducción al Análisis de Datos · UTN FRA · División 343**  
**Integrantes:** Matías, Julián y Emmanuel. **Responsable de esta tarea:** Matías.

Visualizamos la matriz de Pearson del punto 11 para comparar antigüedad, cargo mensual y cargo acumulado. Exportamos el gráfico en `reports/figures/18_heatmap_correlacion.png`.

## Cómo ejecutarlo

- **Local:** instalá `requirements.txt`, abrí el notebook desde el repositorio y ejecutá todas las celdas.
- **Colab:** abrí el botón o subí este `.ipynb` y ejecutá todas las celdas. El CSV se lee desde GitHub y el PNG queda en el panel **Archivos**.


## 1. Estilo común

Reutilizamos la celda del punto 16, que aplica la tipografía y la paleta del equipo y configura la exportación PNG a 300 dpi.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import seaborn as sns
from matplotlib.ticker import PercentFormatter

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

if EN_COLAB:
    RAIZ = Path.cwd()
else:
    actual = Path.cwd().resolve()
    RAIZ = next(
        (ruta for ruta in (actual, *actual.parents) if (ruta / 'PLAN_TAREAS.md').is_file()),
        None,
    )
    if RAIZ is None:
        raise FileNotFoundError('Abrí Jupyter dentro del repositorio del TPI.')

TAMANO_FIGURA = (9, 5)
DPI_EXPORTACION = 300
PALETA_CATEGORICA = ['#0072B2', '#D55E00', '#009E73', '#CC79A7', '#E69F00', '#56B4E9']
COLORES_CHURN = {'No': '#0072B2', 'Yes': '#D55E00'}
MAPA_CORRELACION = 'vlag'
CARPETA_FIGURAS = RAIZ / 'reports' / 'figures'

def aplicar_estilo():
    """Aplica el estilo común del TPI a las figuras que se creen después."""
    sns.set_theme(
        style='whitegrid',
        context='notebook',
        font='DejaVu Sans',
        palette=PALETA_CATEGORICA,
        rc={
            'figure.figsize': TAMANO_FIGURA,
            'figure.dpi': 110,
            'figure.facecolor': 'white',
            'axes.facecolor': 'white',
            'axes.titlesize': 14,
            'axes.titleweight': 'bold',
            'axes.labelsize': 12,
            'xtick.labelsize': 11,
            'ytick.labelsize': 11,
            'legend.fontsize': 11,
            'axes.spines.top': False,
            'axes.spines.right': False,
            'grid.alpha': 0.25,
            'savefig.dpi': DPI_EXPORTACION,
            'savefig.facecolor': 'white',
        },
    )

def guardar_figura(fig, nombre):
    """Guarda fig en reports/figures/<nombre>.png a 300 dpi y devuelve Path.

    Usar un nombre sin ruta ni extensión, como '17_histograma_tenure'.
    Un nombre existente se sobrescribe para permitir regenerar el gráfico.
    Guardar antes de plt.show() y cerrar luego con plt.close(fig).
    """
    if not nombre or Path(nombre).name != nombre or Path(nombre).suffix:
        raise ValueError('Usá un nombre sin ruta ni extensión.')
    CARPETA_FIGURAS.mkdir(parents=True, exist_ok=True)
    ruta = CARPETA_FIGURAS / f'{nombre}.png'
    fig.savefig(ruta, dpi=DPI_EXPORTACION, bbox_inches='tight', facecolor='white')
    return ruta

aplicar_estilo()
print('Estilo común aplicado. Exportación PNG a 300 dpi en reports/figures/.')


## 2. Carga de datos

Leemos las mismas tres columnas del punto 11. Si el dataset limpio todavía no está disponible, usamos el provisorio. La figura indica cuál fue la fuente.

In [ ]:
from pathlib import Path
from urllib.error import HTTPError

import pandas as pd

try:
    import google.colab
except ImportError:
    EN_COLAB = False
else:
    EN_COLAB = True

variables = ["tenure", "MonthlyCharges", "TotalCharges"]
ruta_limpia = Path("data/processed/telco_clean.csv")
ruta_provisoria = Path("data/processed/telco_provisorio.csv")

if EN_COLAB:
    url_repo = "https://raw.githubusercontent.com/julianschmidt1/tpi-analisis-de-datos/main/"
    fuente = url_repo + ruta_limpia.as_posix()
    try:
        datos = pd.read_csv(fuente, usecols=variables)
    except HTTPError as error:
        if error.code != 404:
            raise
        fuente = url_repo + ruta_provisoria.as_posix()
        datos = pd.read_csv(fuente, usecols=variables)
else:
    actual = Path.cwd().resolve()
    raiz = next(
        (ruta for ruta in (actual, *actual.parents)
         if (ruta / ruta_limpia).is_file() or (ruta / ruta_provisoria).is_file()),
        None,
    )
    if raiz is None:
        raise FileNotFoundError("Falta el CSV de la tarea 3 o el dataset limpio de la tarea 9.")
    fuente = raiz / ruta_limpia
    if not fuente.is_file():
        fuente = raiz / ruta_provisoria
    datos = pd.read_csv(fuente, usecols=variables)

datos = datos[variables]
print(f"Fuente: {fuente}")

## 3. Heatmap de Pearson

Calculamos Pearson con el mismo criterio del punto 11. Cada celda muestra el coeficiente con tres decimales. Usamos una escala divergente de −1 a 1 centrada en cero para distinguir el signo y la intensidad de la asociación. La diagonal vale 1 porque compara cada variable consigo misma.

In [ ]:
pearson = datos.corr(method="pearson")
etiquetas = ["Antigüedad", "Cargo mensual", "Cargo acumulado"]

fig, ax = plt.subplots(layout="constrained")
sns.heatmap(
    pearson, annot=True, fmt=".3f", cmap=MAPA_CORRELACION,
    vmin=-1, vmax=1, center=0, square=True, linewidths=0.5,
    xticklabels=etiquetas, yticklabels=etiquetas,
    cbar_kws={"label": "Coeficiente de Pearson"}, ax=ax,
)
ax.set_title("Correlación entre antigüedad y cargos")
ax.tick_params(axis="x", labelrotation=0)
ax.tick_params(axis="y", labelrotation=0)
nombre_fuente = str(fuente).rsplit("/", 1)[-1] if EN_COLAB else fuente.name
fig.supxlabel(f"Fuente: {nombre_fuente}", fontsize=9)

ruta_figura = guardar_figura(fig, "18_heatmap_correlacion")
print(f"Figura exportada: {ruta_figura.relative_to(RAIZ).as_posix()}")
plt.show()
plt.close(fig)

## 4. Interpretación con el CSV limpio

La celda de mayor correlación fuera de la diagonal une **antigüedad y cargo acumulado (0,826)**. Quienes llevan más meses en la empresa tienden a acumular más cargos. El cargo mensual también se asocia con el acumulado (0,651). La asociación entre antigüedad y cargo mensual es más débil (0,248), así que los meses de servicio dan poca información sobre cuánto paga un cliente cada mes.

Con `telco_clean.csv`, las tres comparaciones usan los 7.043 clientes.